## Compare feature-only & text-only model

In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

prediction_dir = project_root / "results/predictions"

tabular_predictions = pd.read_csv(
    prediction_dir / "tabular_validation_predictions.csv"
)

text_predictions = pd.read_csv(
    prediction_dir / "text_validation_predictions.csv"
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: /Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor


In [3]:
comparison = tabular_predictions.merge(
    text_predictions[
        [
            "id",
            "flop",
            "text_probability",
            "text_prediction",
        ]
    ],
    on="id",
    how="inner",
    suffixes=("_tabular", "_text"),
    validate="one_to_one",
)

if not (
    comparison["flop_tabular"]
    == comparison["flop_text"]
).all():
    raise ValueError("True labels do not match between prediction files.")

comparison = comparison.rename(
    columns={"flop_tabular": "flop"}
).drop(columns="flop_text")

print("Compared movies:", len(comparison))
display(comparison.head())

Compared movies: 185


,id,title,release_year,flop,tabular_probability,tabular_prediction,text_probability,text_prediction
0,20504,The Book of Eli,2010,1,0.535522,1,0.514579,1
1,12201,Edge of Darkness,2010,1,0.500518,1,0.507329,1
2,32657,Percy Jackson & the Olympians: The Lightning T...,2010,1,0.485389,0,0.502778,1
3,26389,From Paris with Love,2010,1,0.533839,1,0.508984,1
4,32856,Valentine's Day,2010,0,0.495174,0,0.495263,0


In [4]:
comparison["tabular_correct"] = (
    comparison["tabular_prediction"] == comparison["flop"]
)

comparison["text_correct"] = (
    comparison["text_prediction"] == comparison["flop"]
)

comparison_counts = {
    "Both correct": int(
        (
            comparison["tabular_correct"]
            & comparison["text_correct"]
        ).sum()
    ),
    "Tabular only correct": int(
        (
            comparison["tabular_correct"]
            & ~comparison["text_correct"]
        ).sum()
    ),
    "Text only correct": int(
        (
            ~comparison["tabular_correct"]
            & comparison["text_correct"]
        ).sum()
    ),
    "Both wrong": int(
        (
            ~comparison["tabular_correct"]
            & ~comparison["text_correct"]
        ).sum()
    ),
}

comparison_counts

{'Both correct': 59,
 'Tabular only correct': 43,
 'Text only correct': 42,
 'Both wrong': 41}

In [5]:
prediction_disagreement = (
    comparison["tabular_prediction"]
    != comparison["text_prediction"]
).mean()

print(
    f"Prediction disagreement: "
    f"{prediction_disagreement:.1%}"
)

Prediction disagreement: 45.9%


In [6]:
probability_correlation = comparison[
    ["tabular_probability", "text_probability"]
].corr().iloc[0, 1]

print(
    f"Probability correlation: "
    f"{probability_correlation:.3f}"
)

Probability correlation: 0.014


In [8]:
comparison["average_probability"] = (
    comparison["tabular_probability"]
    + comparison["text_probability"]
) / 2

comparison["average_prediction"] = (
    comparison["average_probability"] >= 0.5
).astype(int)

In [13]:
from src.evaluation import evaluate_classifier
ensemble_metrics = evaluate_classifier(
    y_true=comparison["flop"],
    y_pred=comparison["average_prediction"],
    y_prob=comparison["average_probability"])
ensemble_metrics

{'accuracy': 0.5621621621621622,
 'balanced_accuracy': 0.561711079943899,
 'flop_precision': 0.5555555555555556,
 'flop_recall': 0.6451612903225806,
 'flop_f1': 0.5970149253731343,
 'roc_auc': 0.5614773258532024}